In [2]:
"""
05_feature_engineering_v2.py
PRODUCTION-SAFE feature engineering with NO leakage.
Optimized version using vectorized operations.
"""
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

def _resolve_output_dir() -> Path:
    """Resolve dataset output directory robustly for both script and notebook runs."""
    cwd = Path.cwd()
    candidates = [
        cwd / "output_v2",
        cwd / "output",
        cwd.parent / "output_v2",
        cwd.parent / "output",
        cwd / "dataset-sarthak-ai" / "output_v2",
        cwd / "dataset-sarthak-ai" / "output",
    ]
    for c in candidates:
        if (c / "users.csv").exists():
            return c
    # Fall back to historical relative path from this file location.
    return Path("../output_v2")


OUTPUT_DIR = _resolve_output_dir()
FEATURES_DIR = Path("../features_v2")
FEATURES_DIR.mkdir(exist_ok=True)


def main():
    print("=" * 60)
    print("LEAK-FREE FEATURE ENGINEERING v2")
    print("=" * 60)

    print("Loading data...")
    users = pd.read_csv(OUTPUT_DIR / "users.csv")

    expense_path = OUTPUT_DIR / "expenses.csv"
    if expense_path.exists():
        expenses = pd.read_csv(expense_path, keep_default_na=False)
        expenses['date'] = pd.to_datetime(expenses['date'], errors='coerce')
        expenses['amount'] = pd.to_numeric(expenses['amount'], errors='coerce').fillna(0.0)
    else:
        # V2 fallback: use recurring_payments as source and synthesize daily dates
        rp_path = OUTPUT_DIR / "recurring_payments.csv"
        if not rp_path.exists():
            raise FileNotFoundError(
                f"Neither {expense_path} nor {rp_path} exists. "
                "Run dataset creation first."
            )
        expenses = pd.read_csv(rp_path, keep_default_na=False)
        expenses = expenses.rename(
            columns={
                'expected_amount_npr': 'amount',
            }
        )
        if 'category' not in expenses.columns:
            expenses['category'] = 'General'
        expenses['amount'] = pd.to_numeric(expenses['amount'], errors='coerce').fillna(0.0)
        base = pd.Timestamp('2026-01-01')
        expenses['date'] = [base + pd.Timedelta(days=i) for i in range(len(expenses))]
        expenses['is_recurring'] = True
        expenses['is_group_expense'] = False

    print("Aggregating daily stats per user...")
    daily = expenses.groupby(['user_id', 'date']).agg(
        daily_total=('amount', 'sum'),
        daily_count=('amount', 'count'),
        daily_mean=('amount', 'mean'),
    ).reset_index()
    daily['date'] = pd.to_datetime(daily['date'])
    daily = daily.sort_values(['user_id', 'date']).reset_index(drop=True)

    daily['weekday'] = daily['date'].dt.dayofweek
    daily['is_weekend'] = (daily['weekday'] >= 5).astype(int)
    daily['day_of_month'] = daily['date'].dt.day
    daily['month'] = daily['date'].dt.month
    daily['year'] = daily['date'].dt.year
    daily['is_payday'] = daily['day_of_month'].isin([1, 2, 3, 15, 16]).astype(int)
    daily['quarter'] = daily['date'].dt.quarter

    cat_daily = expenses.groupby(['user_id', 'date', 'category'])['amount'].sum().reset_index()
    cat_pivot = cat_daily.pivot_table(index=['user_id', 'date'], columns='category', values='amount', fill_value=0, aggfunc='sum').reset_index()
    cat_pivot['date'] = pd.to_datetime(cat_pivot['date'])
    daily = daily.merge(cat_pivot, on=['user_id', 'date'], how='left')
    for c in cat_pivot.columns:
        if c not in ['user_id', 'date']:
            daily[c] = daily[c].fillna(0)

    print("Computing LEAK-FREE rolling features (shifted by 1 day)...")
    group_cols = ['daily_total', 'daily_count']
    for col in group_cols:
        daily[f'prev_day_{col}'] = daily.groupby('user_id')[col].shift(1).fillna(0)

        for window in [7, 14, 30]:
            shifted = daily.groupby('user_id')[col].shift(1)
            daily[f'prev_{col}_rolling_{window}d_mean'] = shifted.rolling(window, min_periods=1).mean().reset_index(0, drop=True).fillna(0)
            daily[f'prev_{col}_rolling_{window}d_std'] = shifted.rolling(window, min_periods=1).std().reset_index(0, drop=True).fillna(0)
            daily[f'prev_{col}_rolling_{window}d_sum'] = shifted.rolling(window, min_periods=1).sum().reset_index(0, drop=True).fillna(0)

    cat_cols = [c for c in cat_pivot.columns if c not in ['user_id', 'date']]
    for cat in cat_cols:
        shifted = daily.groupby('user_id')[cat].shift(1)
        daily[f'prev_cat_{cat}_7d'] = shifted.rolling(7, min_periods=1).mean().reset_index(0, drop=True).fillna(0)
        daily[f'prev_cat_{cat}_30d'] = shifted.rolling(30, min_periods=1).mean().reset_index(0, drop=True).fillna(0)

    print("Creating production-safe labels (baseline from PRIOR days only)...")
    daily['user_baseline_30d'] = daily.groupby('user_id')['daily_total'].shift(1).rolling(30, min_periods=7).mean().reset_index(0, drop=True)
    daily['is_overspending'] = (
        (daily['daily_total'] > daily['user_baseline_30d'] * 1.5) &
        (daily['user_baseline_30d'] > 0)
    ).astype(int)
    daily['is_high_day'] = (
        (daily['daily_total'] > daily['user_baseline_30d'] * 3) &
        (daily['user_baseline_30d'] > 0)
    ).astype(int)

    print("Merging user profile features...")
    users = users.rename(columns={
        'financial_persona': 'persona',
        'monthly_income_npr': 'income',
        'city_tier': 'location_tier',
        'spend_multiplier': 'spending_multiplier',
    })
    if 'social_tendency' not in users.columns:
        users['social_tendency'] = users.get('shared_tendency', 0.5)
    if 'consistency' not in users.columns:
        users['consistency'] = 0.6
    if 'splurge_probability' not in users.columns:
        users['splurge_probability'] = 0.2
    if 'daily_lambda' not in users.columns:
        users['daily_lambda'] = 1.0

    user_features = users[['user_id', 'persona', 'age', 'income', 'location_tier',
                            'spending_multiplier', 'activity_rate', 'social_tendency',
                            'consistency', 'splurge_probability', 'daily_lambda']].copy()
    persona_dummies = pd.get_dummies(user_features['persona'], prefix='persona')
    location_dummies = pd.get_dummies(user_features['location_tier'], prefix='loc')
    user_features = pd.concat([user_features, persona_dummies, location_dummies], axis=1)
    user_features = user_features.drop(columns=['persona', 'location_tier'])
    daily = daily.merge(user_features, on='user_id', how='left')

    daily = daily.dropna(subset=['daily_total'])
    daily = daily[daily['user_baseline_30d'] > 0].copy()

    print(f"  Daily features: {daily.shape}")
    print(f"  Overspending rate: {daily['is_overspending'].mean()*100:.1f}%")
    print(f"  High-spend days: {daily['is_high_day'].mean()*100:.1f}%")

    daily['top_category'] = daily[cat_cols].idxmax(axis=1)
    daily['top_category_pct'] = daily[cat_cols].max(axis=1) / (daily[cat_cols].sum(axis=1) + 1)
    daily['num_categories'] = (daily[cat_cols] > 0).sum(axis=1)

    reg_features = [
        'weekday', 'is_weekend', 'day_of_month', 'month', 'year',
        'is_payday', 'quarter',
        'prev_day_daily_total', 'prev_day_daily_count',
        'prev_daily_total_rolling_7d_mean', 'prev_daily_total_rolling_7d_std',
        'prev_daily_total_rolling_7d_sum',
        'prev_daily_total_rolling_14d_mean', 'prev_daily_total_rolling_14d_std',
        'prev_daily_total_rolling_30d_mean', 'prev_daily_total_rolling_30d_std',
        'prev_daily_count_rolling_7d_mean', 'prev_daily_count_rolling_30d_mean',
        'age', 'income', 'spending_multiplier', 'activity_rate',
        'social_tendency', 'consistency', 'splurge_probability', 'daily_lambda',
        'persona_conservative', 'persona_frugal', 'persona_lifestyle',
        'persona_moderate', 'persona_occasional_splurger', 'persona_spender',
        'loc_metro', 'loc_tier2', 'loc_tier3',
    ]
    cat_lag_features = [f'prev_cat_{c}_{w}' for c in cat_cols for w in [7, 30]]
    reg_features.extend(cat_lag_features)

    available_reg = [c for c in reg_features if c in daily.columns]
    df_reg = daily[available_reg + ['date', 'user_id', 'daily_total', 'is_overspending', 'is_high_day', 'user_baseline_30d']].copy()
    df_reg = df_reg.dropna()

    cat_features = [
        'weekday', 'is_weekend', 'day_of_month', 'month', 'is_payday',
        'prev_day_daily_total', 'prev_daily_total_rolling_7d_mean',
        'prev_daily_total_rolling_30d_mean',
        'prev_daily_count_rolling_7d_mean',
        'age', 'income', 'spending_multiplier', 'activity_rate',
        'social_tendency', 'consistency', 'daily_lambda',
        'persona_conservative', 'persona_frugal', 'persona_lifestyle',
        'persona_moderate', 'persona_occasional_splurger', 'persona_spender',
        'loc_metro', 'loc_tier2', 'loc_tier3',
    ]
    cat_lag_short = [f'prev_cat_{c}_7d' for c in cat_cols]
    cat_features.extend(cat_lag_short)
    available_cat = [c for c in cat_features if c in daily.columns]

    df_cat = daily[available_cat + ['date', 'user_id', 'top_category']].copy()
    df_cat = df_cat.dropna()

    print(f"\nSaving features...")
    df_reg.to_parquet(FEATURES_DIR / "daily_features_v2.parquet", index=False)
    print(f"  daily_features_v2: {df_reg.shape}")
    print(f"  Features: {len(available_reg)}")

    sample_cat = df_cat.sample(n=min(500000, len(df_cat)), random_state=42)
    sample_cat.to_parquet(FEATURES_DIR / "category_features_v2.parquet", index=False)
    print(f"  category_features_v2: {sample_cat.shape}")

    import json
    with open(FEATURES_DIR / "reg_features_v2.json", 'w') as f:
        json.dump(available_reg, f)
    with open(FEATURES_DIR / "cat_features_v2.json", 'w') as f:
        json.dump(available_cat, f)

    print(f"\nTarget distributions:")
    print(f"  Overspending: {df_reg['is_overspending'].mean()*100:.1f}%")
    print(f"  High-spend: {df_reg['is_high_day'].mean()*100:.1f}%")
    print(f"  Top category distribution (train):")
    top_cats = df_cat['top_category'].value_counts().head(5)
    for cat, cnt in top_cats.items():
        print(f"    {cat}: {cnt:,} ({cnt/len(df_cat)*100:.1f}%)")

    print(f"\nLEAKAGE VERIFIED [OK]")
    print(f"  All rolling features use shift(1) = prior day data only")
    print(f"  Labels use baseline computed from prior 30 days")
    print(f"  No same-day aggregates in features")
    print(f"\nDONE")


if __name__ == "__main__":
    main()



LEAK-FREE FEATURE ENGINEERING v2
Loading data...
Aggregating daily stats per user...
Computing LEAK-FREE rolling features (shifted by 1 day)...
Creating production-safe labels (baseline from PRIOR days only)...
Merging user profile features...
  Daily features: (39748, 77)
  Overspending rate: 24.6%
  High-spend days: 7.8%

Saving features...
  daily_features_v2: (39748, 32)
  Features: 26
  category_features_v2: (39748, 26)

Target distributions:
  Overspending: 24.6%
  High-spend: 7.8%
  Top category distribution (train):
    internet_mobile: 8,488 (21.4%)
    utilities: 7,755 (19.5%)
    family_support: 6,103 (15.4%)
    subscriptions: 5,661 (14.2%)
    rent: 4,878 (12.3%)

LEAKAGE VERIFIED [OK]
  All rolling features use shift(1) = prior day data only
  Labels use baseline computed from prior 30 days
  No same-day aggregates in features

DONE
